# Crescendo Defense: Multi-Turn Jailbreak Protection

This notebook demonstrates how the **crescendo-defense** framework protects `Llama-3.2-3B-Instruct` against Crescendo-style multi-turn alignment attacks. 

We will:
1. Initialize the system config and subcomponents.
2. Simulate a multi-turn attack scenario (e.g., Lockpicking bypass instructions).
3. Monitor how threat indices and risk velocities escalate over time.
4. Compare the mitigation profiles of **Strategy A (Context Condensation)** vs **Strategy B (Backtracking & Perturbation)**.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np

# Add the parent folder to access local packages
sys.path.append(os.path.abspath(".."))

from config import Config
from attacks.attack_vectors import CrescendoAttackSimulator
from pipeline.defense_pipeline import DefensePipeline
from benchmark.metrics import Evaluator

print("crescendo-defense modules imported successfully.")

## 1. Simulate Threat Escalation (No Defense)
First, we run the attack sequence without any active mitigations to see the raw threat curve escalation.

In [ ]:
# Load clean configs
config = Config()
simulator = CrescendoAttackSimulator()

# Let's load the lockpicking scenario
scenario_prompts = simulator.generate_conversation("lockpicking", is_attack=True)

# We will use a baseline detector directly to observe raw scores
from detector.escalation_detector import EscalationDetector
raw_detector = EscalationDetector(config)

results = []
for idx, turn in enumerate(scenario_prompts):
    analysis = raw_detector.analyze_turn(turn)
    results.append({
        "Turn": idx + 1,
        "Prompt": turn[:60] + "...",
        "Threat Score": analysis["current_threat_score"],
        "Toxicity": analysis["toxicity_score"],
        "Semantic Similarity": analysis["semantic_similarity_score"],
        "Escalation Rate": analysis["escalation_rate"]
    })

df_raw = pd.DataFrame(results)
df_raw

## 2. Evaluate Strategy A: Context Condensation
We process the turns through the `DefensePipeline` using **Strategy A**.

In [ ]:
pipeline_a = DefensePipeline(config, strategy_mode="strategy_a", use_mock_model=True)
pipeline_a.reset_session()

history = []
results_a = []

for idx, turn in enumerate(scenario_prompts):
    response, telemetry = pipeline_a.process_turn(history, turn)
    results_a.append({
        "Turn": idx + 1,
        "Threat": telemetry["threat_score"],
        "Intervened": telemetry["defense_triggered"],
        "Strategy": telemetry["strategy_applied"],
        "Response": response[:100] + "..."
    })
    history.append({"role": "user", "content": turn})
    history.append({"role": "assistant", "content": response})

df_a = pd.DataFrame(results_a)
df_a

## 3. Evaluate Strategy B: Historical Backtracking
We run the exact same attack scenario, but switch the mitigation pipeline to **Strategy B**.

In [ ]:
pipeline_b = DefensePipeline(config, strategy_mode="strategy_b", use_mock_model=True)
pipeline_b.reset_session()

history = []
results_b = []

for idx, turn in enumerate(scenario_prompts):
    response, telemetry = pipeline_b.process_turn(history, turn)
    results_b.append({
        "Turn": idx + 1,
        "Threat": telemetry["threat_score"],
        "Intervened": telemetry["defense_triggered"],
        "Strategy": telemetry["strategy_applied"],
        "Backtracked": telemetry["backtracked"],
        "Response": response[:100] + "..."
    })
    # Note: Strategy B may backtrack and trim history, but we simulate standard interaction logs
    history.append({"role": "user", "content": turn})
    history.append({"role": "assistant", "content": response})

df_b = pd.DataFrame(results_b)
df_b

## 4. Compare Threat Profiles & Safe Intervention Outcomes

In [ ]:
import matplotlib.pyplot as plt
try:
    plt.figure(figsize=(10, 5))
    plt.plot(df_raw["Turn"], df_raw["Threat Score"], marker='o', label='Raw Threat Score (No Defense)', color='crimson', linewidth=2)
    plt.plot(df_a["Turn"], df_a["Threat"], marker='s', label='Threat Under Strategy A (Condensation)', color='teal', linestyle='--')
    plt.plot(df_b["Turn"], df_b["Threat"], marker='^', label='Threat Under Strategy B (Backtracking)', color='indigo', linestyle=':')
    
    plt.title("Threat Score Progression Across Attack Turns", fontsize=14)
    plt.xlabel("Turn Index", fontsize=12)
    plt.ylabel("Observed Risk / Threat Score", fontsize=12)
    plt.xticks(df_raw["Turn"])
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
except ImportError:
    print("matplotlib not installed. Text summary representation:")
    print(f"Raw Threat values:      {list(df_raw['Threat Score'].round(3))}")
    print(f"Strategy A Threat values: {list(df_a['Threat'].round(3))}")
    print(f"Strategy B Threat values: {list(df_b['Threat'].round(3))}")